# Séance 2 — Codes polyalphabétiques : Vigenère et son attaque

**Cours de référence** : P. Boyer, *Cryptographie 2013*, fin du chapitre 2
(« Codes polyalphabétiques »).

## Objectifs de la séance

1. mettre en œuvre le **code de Vigenère** (une substitution différente à chaque lettre) ;
2. comprendre **pourquoi l'analyse de fréquences globale échoue** sur ce chiffre
   (expérience à faire soi-même dans ce notebook) ;
3. comprendre la **faiblesse structurelle** qui le fait tomber : la clef se répète ;
4. casser les deux messages interceptés (la période est donnée).

## Ce que vous rendez

Votre fichier `tp2_vigenere.py` complété (douze fonctions) et les réponses aux
questions de rapport.

    python3 test_tp2_vigenere.py


## 1. Rappels du cours

Dans un code de substitution simple, la même lettre du clair devient toujours la même
lettre du chiffré. Vigenère change de règle à chaque lettre :

$$c_i = \big(\text{rang}(m_i) + \text{rang}(k_{i \bmod \ell})\big) \bmod 26$$

où $\ell$ est la longueur de la clef, appelée **période**.

### La convention de ce TP (importante)

* les espaces et la ponctuation sont recopiés tels quels et **ne consomment pas** de
  lettre de clef ;
* la clef se répète si elle est plus courte que le message ;
* le déchiffrement est la même formule avec $-\text{rang}(k)$.

Vigenère est resté célèbre sous le nom de « chiffre indéchiffrable » — surnom excessif,
comme vous allez le montrer. Il a fallu deux siècles (Babbage, Kasiski, Friedman) pour
le casser.


In [ ]:
import os, sys, subprocess

def trouver_racine(depart):
    """Remonte les dossiers jusqu'à trouver la racine du dépôt."""
    while True:
        if os.path.exists(os.path.join(depart, "outils", "reference.py")):
            return depart
        parent = os.path.dirname(depart)
        if parent == depart:
            raise RuntimeError("racine du dépôt introuvable")
        depart = parent

RACINE = trouver_racine(os.getcwd())
if RACINE not in sys.path:
    sys.path.insert(0, RACINE)

def trouver(nom):
    """Retrouve un fichier du dépôt (son dossier de séance, peu importe où l'on est)."""
    for base, _, fichiers in os.walk(RACINE):
        if nom in fichiers:
            return os.path.join(base, nom)
    raise FileNotFoundError(nom)

DOSSIER = os.path.dirname(trouver("test_tp2_vigenere.py"))

def verifier():
    """Lance les épreuves automatiques de la séance et affiche le bilan."""
    resultat = subprocess.run([sys.executable, "test_tp2_vigenere.py"], cwd=DOSSIER,
                              capture_output=True, text=True)
    print(resultat.stdout.strip()[-2500:])

def executer(consigne, fonction):
    """Exécute une cellule d'exercice, avec un rappel si votre fonction manque encore."""
    try:
        return fonction()
    except NotImplementedError as erreur:
        print("Cette cellule attend une fonction de votre fichier :", consigne)
        print("   ->", erreur)
        print("   Écrivez la fonction, enregistrez le fichier .py, puis relancez la cellule.")
        return None

def charger():
    """Recharge votre fichier de TP (après l'avoir modifié, relancez cette cellule)."""
    import importlib
    if DOSSIER not in sys.path:
        sys.path.insert(0, DOSSIER)
    if "tp2_vigenere" in sys.modules:
        return importlib.reload(sys.modules["tp2_vigenere"])
    import importlib.util
    spec = importlib.util.spec_from_file_location("tp2_vigenere", os.path.join(DOSSIER, "tp2_vigenere.py"))
    module = importlib.util.module_from_spec(spec)
    sys.modules["tp2_vigenere"] = module
    spec.loader.exec_module(module)
    return module

print("Racine du dépôt :", RACINE)
print("Dossier de la séance :", DOSSIER)


## 2. Mettre en œuvre Vigenère (fonctions A.1 à B.2)

Écrivez dans `tp2_vigenere.py` : `indice_de_la_lettre`, `decaler_lettre`,
`vider_les_espaces`, `vigenere_chiffrer`, `vigenere_dechiffrer`, puis vérifiez.


In [ ]:
tp = charger()

def _exercice():
    print("chiffrement :", tp.vigenere_chiffrer("BONJOUR LE MONDE", "CLE"))
    print("déchiffrement :", tp.vigenere_dechiffrer("DZRLZYT WI OZRFP", "CLE"))

executer("A.1 à B.2", _exercice)
verifier()

## 3. Expérience centrale : les fréquences survivent-elles ?

C'est toute la question. Comparez l'histogramme :

1. d'un texte français ;
2. de ce texte chiffré par **César** ;
3. du même texte chiffré par **Vigenère**.


In [ ]:
from outils.texte_francais import barres, frequences
from outils.reference import cesar_chiffrer, vigenere_chiffrer

clair = ("LE MESSAGE ARRIVE CE SOIR A DIX HEURES DEVANT LA PORTE DU JARDIN PRES DE LA FONTAINE "
         "LES ENFANTS JOUENT DANS LA COUR DE L ECOLE ET LE MAITRE FERME LES VOLETS")

for titre, texte in (("CLAIR (français)", clair),
                     ("CÉSAR, clef 7", cesar_chiffrer(clair, 7)),
                     ("VIGENÈRE, clef SECRET", vigenere_chiffrer(clair, "SECRET"))):
    print("=" * 70)
    print(titre)
    print("=" * 70)
    print(barres(frequences(texte), largeur=30))

**Observation** : avec César, l'histogramme garde exactement la même forme, simplement
translatée — le E reste la lettre dominante. Avec Vigenère, les lettres sont beaucoup
plus régulières : le E du clair devient E, C, W, H... selon la position. L'analyse de
fréquences globale ne dit plus rien.

Mais alors, comment casser Vigenère ? C'est la question de la partie C.


## 4. L'idée de l'attaque : découper en colonnes

Si la clef fait $\ell$ lettres, alors :

* les caractères aux positions $0, \ell, 2\ell, \dots$ sont tous décalés **de la même quantité** ;
* idem pour les positions $1, 1+\ell, \dots$, etc.

Autrement dit, en découpant le message en $\ell$ colonnes, **chaque colonne est un simple
code de César** — donc cassable par analyse de fréquences. La seule inconnue qui reste
est $\ell$.

### Expérience

Prenez un message chiffré par Vigenère avec la clef `SECRET` (période 6). Calculez
l'histogramme de chacune des 6 colonnes : chacune ressemble à du français décalé !


In [ ]:
from outils.reference import vigenere_chiffrer

clair = ("LE MESSAGE ARRIVE CE SOIR A DIX HEURES DEVANT LA PORTE DU JARDIN PRES DE LA FONTAINE "
         "LES ENFANTS JOUENT DANS LA COUR DE L ECOLE ET LE MAITRE FERME LES VOLETS")
chiffre = vigenere_chiffrer(clair, "SECRET")
lettres = "".join(c for c in chiffre if "A" <= c <= "Z")
colonnes = [lettres[i::6] for i in range(6)]
for numero, colonne in enumerate(colonnes):
    print(f"--- colonne {numero} : {colonne[:45]}...")
    print("   ", barres(frequences(colonne), largeur=20).replace("\n", "\n    ")[:400])

## 5. Casser les messages interceptés (partie C)

Écrivez maintenant `forcer_cesar_sur_colonne`, `colonnes`, `recoller`, puis
`forcer_vigenere`, et attaquez les deux dépêches `MESSAGE_VIGENERE_1` et
`MESSAGE_VIGENERE_2` (leurs périodes sont données : `PERIODE_1` et `PERIODE_2`).


In [ ]:
tp = charger()

def _exercice():
    for nom, message, periode in (("Dépêche 1", tp.MESSAGE_VIGENERE_1, tp.PERIODE_1),
                                  ("Dépêche 2", tp.MESSAGE_VIGENERE_2, tp.PERIODE_2)):
        cle, clair = tp.forcer_vigenere(message, periode)
        print("=" * 70)
        print(f"{nom} — longueur {len(message)}, période {periode}")
        print("Clef retrouvée :", cle)
        print(clair)

executer("C.1 à C.4 forcer_vigenere", _exercice)
verifier()

## 6. L'attaque par texte connu (partie D)

Mettre la main sur un message en clair (un rapport, un communiqué) suffit à retrouver
la clef : on compare les deux textes lettre à lettre. C'est ainsi que Kasiski, officier
prussien, a commencé à casser Vigenère au XIXe siècle. Cette méthode reste aujourd'hui
la base des attaques dites « à texte clair connu » (known-plaintext attacks).

La longueur de la clef se déduit de la **plus petite période** de la suite des décalages
observés. Écrivez `cle_par_texte_connu`, puis expérimentez.


In [ ]:
tp = charger()
from outils.reference import vigenere_chiffrer

def _exercice():
    clair = "LE MESSAGE ARRIVE CE SOIR A DIX HEURES"
    for cle in ("CLE", "VIGENERE", "MATHEMATIQUES"):
        chiffre = vigenere_chiffrer(clair, cle)
        print(f"clef {cle!r} -> clef retrouvée : {tp.cle_par_texte_connu(clair, chiffre)!r}")
    print()
    print("Et si la clef est plus longue que le texte connu ?")
    print("  ", repr(tp.cle_par_texte_connu("BONJOUR", vigenere_chiffrer("BONJOUR", "TRESLONGMOTDECLE"))))

executer("D.1 cle_par_texte_connu", _exercice)
verifier()

## 7. La question des espaces (à traiter dans le rapport)

Deux conventions sont possibles :

* **A** : les espaces et la ponctuation ne consomment pas de lettre de clef (convention
  de ce TP, et des exercices du cours) ;
* **B** : chaque caractère, y compris l'espace, consomme une lettre de clef.

Un cryptoanalyste ne connaît pas la convention utilisée. La cellule suivante chiffre un
message avec la convention B, puis compare le résultat avec la méthode A : elles
n'échouent pas de la même façon avant et après le premier espace !


In [ ]:
clair = "BONJOUR LE MONDE"
cle = "CLE"

def chiffrer_tout_consomme(texte, cle):
    """Convention B : chaque caractère consomme une lettre de clef."""
    chiffre = ""
    for position, caractere in enumerate(texte.upper()):
        if "A" <= caractere <= "Z":
            decalage = ord(cle[position % len(cle)]) - ord("A")
            chiffre += chr((ord(caractere) - 65 + decalage) % 26 + 65)
        else:
            chiffre += caractere
    return chiffre

print("convention A (la nôtre) :", vigenere_chiffrer(clair, cle))
print("convention B            :", chiffrer_tout_consomme(clair, cle))
print()
print("Essayez un texte long sans espaces puis un texte avec espaces,")
print("et regardez jusqu'où les deux conventions coïncident.")

## 8. Questions de rapport

1. Pourquoi l'analyse de fréquences globale échoue-t-elle sur Vigenère ? Appuyez votre
   réponse sur les histogrammes obtenus dans ce notebook.
2. Pourquoi, en revanche, l'analyse de fréquences **par colonne** fonctionne-t-elle ?
3. Combien y a-t-il de clefs de longueur 5 ? Et de longueur 8 ? Comparer le coût de
   l'attaque par colonnes (26 essais par colonne) à celui d'une recherche exhaustive sur
   les clefs.
4. Deux messages sont chiffrés avec la **même clef**. Que peut-on en déduire sans rien
   savoir d'autre ? (Indice : comparez les positions où les lettres chiffrées coïncident ;
   c'est l'attaque dite « de deux messages », et c'est ainsi que le VIC a été
   partiellement compromis en 1914... avant d'être repris par les Britanniques.)
5. Quelle est la faiblesse structurelle de Vigenère ? Que faudrait-il faire pour la
   corriger ? Pourquoi cela ne suffit-il pas encore à faire un chiffre moderne ?
6. Expérience : chiffrez un message avec `CLE`, puis chiffrez le résultat avec `CLE` de
   nouveau. Comparer avec le chiffrement par une seule clef. Que constatez-vous ?


In [ ]:
# Réponse expérimentale à la question 6
clair = "LE MESSAGE ARRIVE CE SOIR"
double = vigenere_chiffrer(vigenere_chiffrer(clair, "CLE"), "CLE")
simple = vigenere_chiffrer(clair, "CLE")
print("clair          :", clair)
print("Vigenère CLE   :", simple)
print("Vigenère 2 x   :", double)
print()
print("Equivalent à un Vigenère avec quelle clef ?")
for cle in ("ABCDEF", "ACEGIK", "CCCCCC", "GGGGGG", "FFFFF"):
    print(f"  {cle!r:10s} :", vigenere_chiffrer(clair, cle))

## 9. Pour aller plus loin (facultatif)

* **Le « compromis de Vigenère »** : et si l'on utilisait une clef aussi longue que le
  message, tirée au hasard (masque jetable, ou *one-time pad*) ? Pourquoi est-ce alors
  incassable, et pourquoi est-ce impraticable dans la vie réelle ? (chapitre des codes
  à confidentialité parfaite)
* **La machine Enigma** : elle réalise une substitution polyalphabétique *sans période
  fixe*, ce qui la rend beaucoup plus solide. Le cours raconte comment les Polonais puis
  Turing l'ont cassée en exploitant une répétition (la clef de message chiffrée deux fois).
